# pLM Covariance Pooling — Run Experiments

**PP1 SoSe2026 | TU München**  
Team: Joel Simon, Julius Schmidt, Lisa Börner, Andreas Weitz

Trains pooler + probe on cached ProtT5 embeddings already in Drive. ProtT5 itself is not loaded here — only the small probe heads (and the PCA / unsupervised poolers) are fit on GPU.

### Workflow
1. GPU check
2. Mount Drive
3. Configuration (Drive paths)
4. Clone + install repo
5. Wire Drive into `data/embeddings/` and `data/raw/`
6. Fit PCA pooler (closed-form, fast)
7. (Optional) Train unsupervised pooler (SGD)
8. Run experiments (all five methods × two tasks)
9. Mirror results back to Drive

## 1 · GPU check

In [ ]:
import torch

if not torch.cuda.is_available():
    print('No GPU detected — probe training will be slow on CPU.\n'
          'Runtime -> Change runtime type -> GPU recommended.')
else:
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU:  {gpu.name}')
    print(f'VRAM: {gpu.total_memory / 1e9:.1f} GB')
    print(f'CUDA: {torch.version.cuda}')

## 2 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print('Top-level folders in MyDrive:')
for f in sorted(os.listdir('/content/drive/MyDrive')):
    print(f'  {f}')

## 3 · Configuration

Set **`DRIVE_ROOT`** to the folder on your Drive that contains the embeddings and (ideally) the label CSVs.

Expected layout under `DRIVE_ROOT`:
```
DRIVE_ROOT/
├── deeploc_train.h5
├── deeploc_test.h5
├── meltome_train.h5
├── meltome_test.h5
└── raw/                  # optional, can also live elsewhere — see RAW_ROOT below
    ├── deeploc/{train,test}_labels.csv
    └── meltome/{train,test}_labels.csv
```
If labels are under a different path, set `RAW_ROOT` separately.

In [ ]:
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/pp1_sop')   # ← embeddings folder
RAW_ROOT   = DRIVE_ROOT / 'raw'                        # ← labels parent dir
RESULTS_OUT = DRIVE_ROOT / 'results'                   # ← runs land here too
# ──────────────────────────────────────────────────────────────────────────

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

expected_h5 = ['deeploc_train.h5', 'deeploc_test.h5',
               'meltome_train.h5', 'meltome_test.h5']
expected_csv = ['deeploc/train_labels.csv', 'deeploc/test_labels.csv',
                'meltome/train_labels.csv', 'meltome/test_labels.csv']

missing = []
print(f'{"File":<45} {"Status"}')
print('-' * 60)
for f in expected_h5:
    p = DRIVE_ROOT / f
    ok = p.exists() and p.stat().st_size > 0
    print(f'{str(p):<45} {"OK" if ok else "MISSING"}')
    if not ok: missing.append(str(p))
for f in expected_csv:
    p = RAW_ROOT / f
    ok = p.exists() and p.stat().st_size > 0
    print(f'{str(p):<45} {"OK" if ok else "MISSING"}')
    if not ok: missing.append(str(p))

if missing:
    print('\nMissing files — fix DRIVE_ROOT/RAW_ROOT or upload them.')
else:
    print('\nAll inputs present.')

## 4 · Clone + install repo

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/Julius-Schmidt/pLM-covariance-pooling.git'
REPO_DIR = Path('/content/pLM-covariance-pooling')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],
    cwd=str(REPO_DIR), check=True,
)
print('Repo installed.')

## 5 · Wire Drive into the repo

Configs reference `data/embeddings/*.h5` and `data/raw/*/`. Symlink those to Drive so the YAMLs work without modification.

In [ ]:
import os, shutil
from pathlib import Path

DATA_DIR = REPO_DIR / 'data'
EMB_LINK = DATA_DIR / 'embeddings'
RAW_LINK = DATA_DIR / 'raw'
MODELS_LINK = REPO_DIR / 'models'
MODELS_DRIVE = DRIVE_ROOT / 'models'
MODELS_DRIVE.mkdir(parents=True, exist_ok=True)

DATA_DIR.mkdir(exist_ok=True)

def relink(link: Path, target: Path):
    if link.is_symlink() or link.exists():
        if link.is_symlink():
            link.unlink()
        else:
            shutil.rmtree(link)
    link.symlink_to(target)
    print(f'{link} -> {target}')

relink(EMB_LINK, DRIVE_ROOT)
relink(RAW_LINK, RAW_ROOT)
relink(MODELS_LINK, MODELS_DRIVE)

# Sanity-check by listing what the YAMLs will see.
for p in ['data/embeddings/deeploc_train.h5',
          'data/raw/deeploc/train_labels.csv',
          'data/embeddings/meltome_train.h5',
          'data/raw/meltome/train_labels.csv']:
    full = REPO_DIR / p
    print(f'{p}: {"OK" if full.exists() else "MISSING"}')

## 6 · Fit PCA pooler (required for cov_pca configs)

Closed-form, one pass over the embeddings — fast. Fits one checkpoint per dc value referenced by the configs.

In [ ]:
import subprocess, sys

DC_VALUES = [8, 16, 24, 32, 48]   # match the dc sweep grid

def stream(cmd, cwd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, cwd=str(cwd))
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed ({proc.returncode}): {" ".join(cmd)}')

for dc in DC_VALUES:
    out = MODELS_LINK / f'pca_pooler_dc{dc}.pt'
    if out.exists():
        print(f'dc={dc}: exists, skipping.')
        continue
    stream([
        sys.executable, str(REPO_DIR / 'scripts' / 'fit_pca_pool.py'),
        '--embeddings', 'data/embeddings/deeploc_train.h5',
        '--d', '1024', '--dc', str(dc),
        '--output', str(out),
    ], cwd=REPO_DIR)

## 7 · (Optional) Train unsupervised pooler

Frobenius autoencoder via SGD. Needed only for `cov_unsupervised` configs. Skip if you only care about mean / cov_supervised / cov_pca / hybrid.

In [ ]:
for dc in DC_VALUES:
    out = MODELS_LINK / f'unsup_pooler_dc{dc}.pt'
    if out.exists():
        print(f'dc={dc}: exists, skipping.')
        continue
    stream([
        sys.executable, str(REPO_DIR / 'scripts' / 'train_unsupervised_pool.py'),
        '--embeddings', 'data/embeddings/deeploc_train.h5',
        '--d', '1024', '--dc', str(dc),
        '--epochs', '5', '--batch-size', '32', '--lr', '1e-3',
        '--device', DEVICE,
        '--output', str(out),
    ], cwd=REPO_DIR)

## 8 · Run experiments

Pick which configs to run. By default: all five methods on both tasks.

In [ ]:
CONFIGS = [
    'configs/scl/mean.yaml',
    'configs/scl/cov_supervised.yaml',
    'configs/scl/cov_pca.yaml',
    'configs/scl/cov_unsupervised.yaml',
    'configs/scl/hybrid.yaml',
    'configs/meltome/mean.yaml',
    'configs/meltome/cov_supervised.yaml',
    'configs/meltome/cov_pca.yaml',
    'configs/meltome/cov_unsupervised.yaml',
    'configs/meltome/hybrid.yaml',
]

# Single-dc run per config. To sweep instead, set DC_SWEEP = [8, 16, 24, 32, 48].
DC_SWEEP = None

for cfg in CONFIGS:
    print(f'\n=== {cfg} ===')
    cmd = [sys.executable, str(REPO_DIR / 'scripts' / 'run_experiment.py'),
           '--config', cfg]
    if DC_SWEEP:
        cmd += ['--dc'] + [str(d) for d in DC_SWEEP]
    stream(cmd, cwd=REPO_DIR)

## 9 · Mirror results to Drive

In [ ]:
import shutil

src = REPO_DIR / 'results'
if not src.exists():
    print(f'No results dir at {src} — nothing to copy.')
else:
    RESULTS_OUT.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in src.rglob('*'):
        if p.is_file():
            rel = p.relative_to(src)
            dst = RESULTS_OUT / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
            n += 1
    print(f'Copied {n} files to {RESULTS_OUT}')

## 10 · Quick look at the summaries

In [ ]:
import json

runs = sorted((REPO_DIR / 'results' / 'runs').glob('*.json'))
for r in runs:
    if r.stem.endswith('_sweep'):
        continue
    d = json.loads(r.read_text())
    metric = 'accuracy_mean' if d['task'] == 'classification' else 'spearman_r_mean'
    std    = metric.replace('_mean', '_std')
    print(f'{d["config"]:<25} {d["method"]:<18} dc={str(d["dc"]):<5} '
          f'{metric.replace("_mean",""):<10}={d[metric]:.4f} ± {d[std]:.4f}')